# Mini Detection
This mini detection algorithm allows you to visually fine-tune detection parameters for each recording

**What you need:** 
1. Each recording consists of one folder with all of the sweeps (.mat files)
2. For blind detection, each recording's condition should be recorded on the metadata csv (see Github for template)

**What you'll get:**
1. A JSON file of all the parameters you used for each cell for future reference
2. The avg mini amplitude per cell recording
3. The mini Hz per cell recording
4. csvs separated by condition and also combined with the information from 2 and 3

# Parameter Tuning

In [ ]:
# Import tuner
import param_tune
from param_tune import ParameterTuner

# Specify folder path with data
parent_folder = ''  # Update this path

# Specify mode (EPSC or IPSC)
tuner = ParameterTuner(parent_folder, mode='EPSC')

# Define parameter sets to test; some examples below. Format: (window_size, sample_step, interval)
parameter_sets = [
    (130, 35, 20),  
    (100, 25, 15),
    (60, 15, 5)
]

# Test parameters on a specific cell
cell_name = "EC1-1"   # Replace with actual cell name

results = tuner.sample_and_test(cell_name, parameter_sets, n_files=3, visualize=True)

In [ ]:
# Save parameters for each cell (add a new line for each cell you want to save parameters for)
tuner.save_parameters("EC1-1", window_size=60, sample_step=15, interval=5, control_pulse_timing='early', mode = 'EPSC')

# Specify output dir
output_dir = ''

# Export parameters to JSON file for later use
tuner.export_parameters("optimized_parameters.json", output_dir)

# Detection

In [ ]:
# Load previously saved parameters from JSON file into the tuner
# Run this instead of re-entering all parameters manually above

import os

json_filename = "optimized_parameters.json"  # update filename as needed
json_path = os.path.join(output_dir, json_filename)

tuner.load_parameters(json_path)

In [ ]:
# Run full detection with optimized parameters
optimized_results = tuner.run_full_detection(output_dir = output_dir)


# Save optimized_results to disk (pickle)
import os
from io_helpers import save_optimized_results

save_path_pkl = os.path.join(output_dir, "optimized_results.pkl")
save_optimized_results(save_path_pkl, optimized_results, use_pickle=True)

In [ ]:
# Load previous optimized results
# from io_helpers import load_optimized_results
# import os 

# load_path_pkl = os.path.join(output_dir, "optimized_results.pkl")
# optimized_results = load_optimized_results(load_path_pkl)

# Group & Export Data

In [ ]:
# Import metadata file
import pandas as pd

metadata = pd.read_csv('metadata.csv') # Change filepath as needed 

# Examine metadata structure
print("Metadata columns:", metadata.columns.tolist())
print("\nFirst few rows of metadata:")
print(metadata.head())
print("\nUnique values in each column:")
for col in metadata.columns:
    print(f"{col}: {metadata[col].unique()}")
print(f"\nMetadata shape: {metadata.shape}")

In [ ]:
# Compute Rs (series resistance), Rm (membrane resistance), and Cm (membrane capacitance)
# Cells with mean Rs > rs_threshold will be excluded from all downstream exports

from functions import export_Rs_Rm_Cm_data
import json

date = "2026XXXX"  # update date as needed

json_path = f"{output_dir}/{date}_optimized_parameters.json"

with open(json_path, 'r') as f:
    parameters = json.load(f)

print(f"Loaded parameters for {len(parameters)} cells from JSON")

# Rs quality control threshold (MΩ) — cells with mean Rs > this value will be excluded
rs_threshold = 25

# Export Rs, Rm, and Cm data with cell-specific control_pulse_timing
Rs_Rm_Cm_df = export_Rs_Rm_Cm_data(
    detected_data_list=optimized_results,
    output_dir=output_dir,
    date=date,
    metadata=metadata,
    cell_id_col='Cell',
    group_col='Condition',
    control_pulse_timing='early',  # Default fallback if not in JSON
    parameters_dict=parameters,    # Contains cell-specific control_pulse_timing
    fs=10000,                      # Sampling frequency
    Rs_threshold=rs_threshold
)

In [ ]:
# Filter optimized_results to only include cells that passed the Rs threshold (≤ rs_threshold MΩ)
from functions import get_cell_names

passing_cells = set(Rs_Rm_Cm_df['Cell_Name'].tolist())
all_cell_names = get_cell_names(optimized_results)
original_count = len(all_cell_names)

optimized_results = [cell for cell, name in zip(optimized_results, all_cell_names) if name in passing_cells]
excluded = [name for name in all_cell_names if name not in passing_cells]

print(f"Rs filter (≤ {rs_threshold} MΩ): kept {len(optimized_results)}/{original_count} cells")
if excluded:
    print(f"Excluded cells: {excluded}")

In [ ]:
import functions

# Separate data by groups defined by metadata
grouped_data, ungrouped = functions.separate_data_by_groups(optimized_results, metadata, 
                                                 cell_id_col='Cell', 
                                                 
                                                 group_col='Condition')

In [ ]:
# Export grouped data to separate CSV files

from functions import export_data, avg_amp_per_cell, hz_per_cell

# Specify date
date = date

# Ensure output_dir has trailing slash
if not output_dir.endswith('/'):
    output_dir = output_dir + '/'

# Prepare data for export
condition_labels = []
amplitude_data = []
frequency_data = []
detection_data_list = []

for group_name, group_results in grouped_data.items():
    condition_labels.append(group_name)
    
    # Calculate amplitudes and frequencies for this group
    group_amplitudes = avg_amp_per_cell(group_results)
    group_frequencies = hz_per_cell(group_results)
    
    amplitude_data.append(group_amplitudes)
    frequency_data.append(group_frequencies)
    detection_data_list.append(group_results)

# Export amplitudes
export_data(
    condition_labels=condition_labels,
    big_data=amplitude_data,
    data_type='Amplitude',
    date=date,
    output_dir=output_dir,
    detected_data_list=detection_data_list
)

# Export frequencies
export_data(
    condition_labels=condition_labels,
    big_data=frequency_data,
    data_type='Frequency',
    date=date,
    output_dir=output_dir,
    detected_data_list=detection_data_list
)

print("\n=== Export Summary ===")
print(f"Exported data for {len(condition_labels)} groups:")
for i, label in enumerate(condition_labels):
    print(f"  {label}: {len(amplitude_data[i])} cells")
print(f"\nFiles saved to: {output_dir}")
print("Created files:")
print("  - Individual group CSV files for amplitudes and frequencies")
print("  - Combined CSV files with all groups together")

# Additional Analyses

In [ ]:
# Extract and export inter-peak interval (IPI) data organized by condition

from functions import export_ipi_by_condition 

# Export with condition-based organization
export_summary = export_ipi_by_condition(
    detected_data_list=optimized_results,
    output_dir=output_dir,
    date=date,
    metadata=metadata,
    cell_id_col='Cell',
    group_col='Condition',
    min_interval_ms=1,
    pad_method='nan'  # or 'drop' or 'repeat_last'
)

In [ ]:
# Extract and export amplitude data organized by condition

from functions import export_amp_by_condition 

# Export with condition-based organization
export_summary = export_amp_by_condition(
    detected_data_list=optimized_results,
    output_dir=output_dir,
    date=date,
    metadata=metadata,
    cell_id_col='Cell',
    group_col='Condition'
)

In [ ]:
# Refresh functions module to pick up any changes made to functions.py
import importlib
import functions
import param_tune
# import reviewer
importlib.reload(functions)
importlib.reload(param_tune)
# importlib.reload(reviewer)
print("Functions module refreshed successfully!")